# Colab Validation Run — Phases 1–6 on Real WOMD Data

Phases 1–6 are committed and tested, but **only against synthetic scenarios built as numpy
arrays** — two vehicles, all 91 timesteps valid, no crowding, no occlusion. Nothing in
Phases 4, 5, or 6 has ever touched a real WOMD shard. `export_shard_geometry` has never
executed at all.

This is a **validation run, not a demo**. The goal is diagnostics: find out where the
synthetic fixtures lied. Every cell below prints its findings even when the news is boring —
"0 interior gaps" is a result, not a skip.

The first run of this notebook measured two Phase 3 bugs on the real shard. Both are now
**fixed** (commits "Fix PET to evaluate both crossing orders" and "Rank Pass 1 on
SDC-restricted TTC/PET instead of all pairs"), and this version re-runs against the same
shard to confirm the fixes hold on real data:

1. **Pass 1 scoring was SDC-agnostic.** `score_scenario` now takes `sdc_index` and ranks on
   TTC/PET restricted to pairs involving the SDC; the old scene-wide values are retained as
   `min_ttc_all_pairs` / `min_pet_all_pairs` diagnostics. Cell 7 shows the saturation
   before/after; cell 7c shows how the ranking moved.
2. **PET evaluated only one crossing order.** `compute_pet_pair` now returns
   `max(enter_b - exit_a, enter_a - exit_b)`, so a negative PET means genuine simultaneous
   occupancy. Cell 7d is a standing regression check on that.

Neither was a correctness bug in Phase 4 — every shipped stress-test result verified its
collision against exact SAT geometry regardless of ranking. This was a prioritisation fix.

> ⚠️ **If you edit any file under `src/` and re-run a cell, Colab does NOT see the change.**
> Python caches the imported module. The fix is **Runtime → Restart session**, then re-run
> from the top — not `del sys.modules[...]`, which is fragile and easy to get subtly wrong
> with submodules.

This notebook writes only to a session-local Postgres database and reads the repo + the
shard. **It never writes back to the repo.**

## 1. Protobuf environment variable

Must be set **before any import**, in its own cell, before anything (including this
notebook's own later cells) imports `google.protobuf` transitively. Setting it after import
has no effect — the C++ implementation is already loaded.

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
print("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION =",
      os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'])

## 2. Installs

`--no-deps` on the Waymo package: it pins an old TensorFlow we do not need (`loader.py`
parses TFRecord with `struct`; only `scenario_pb2` from the Waymo package is used at
runtime). `torch` is deliberately **not installed** — `autograd_optimizer.py` imports it at
module level, but `batch_scorer._stress_one` only imports that module when
`use_autograd=True`, and this notebook runs DE-only, so it is never needed.

Every resolved version is printed so a future failure has a known-good baseline to diff
against.

In [ ]:
!pip install waymo-open-dataset-tf-2-11-0 --no-deps --quiet
!pip install shapely scipy psycopg2-binary --quiet

import importlib.metadata as ilm

for pkg in ["waymo-open-dataset-tf-2-11-0", "shapely", "scipy", "psycopg2-binary",
            "numpy", "protobuf"]:
    try:
        print(f"{pkg:35s} {ilm.version(pkg)}")
    except ilm.PackageNotFoundError:
        print(f"{pkg:35s} NOT INSTALLED")

import sys
print("\npython", sys.version)

## 3. Repo + Drive

Clones the repo if absent, otherwise pulls `main`. Mounts Drive and **asserts the shard file
exists before anything else runs** — failing here, loudly, beats discovering a missing file
300 lines into a batch pass.

`REPO_URL`, `SHARD_PATH` and the run-size knobs (`MAX_SCENARIOS`, `TOP_N`, `DIAG_N`) are the
only things you should need to change to rerun this at a different scale.

In [ ]:
# ── config (the only cell you should need to edit for a different run) ──────
REPO_URL = "https://github.com/AviShrivastava1/av_stress_tester.git"
REPO_DIR = "/content/av_stress_tester"
SHARD_PATH = ("/content/drive/MyDrive/waymo_data/"
              "uncompressed_scenario_training_training.tfrecord-00000-of-01000")

MAX_SCENARIOS = 100   # Pass 1 batch size
TOP_N = 5             # how many scenarios get the Phase 4 stress test (Pass 2)
DIAG_N = 50           # sample size for the 7c/7d rank-correlation + PET experiments
B09_N = 25            # sample size for 10b's B09 before/after. DELIBERATELY NOT
                      # TOP_N: Pass 2 runs at the size the architecture intends,
                      # and TOP_N=5 would leave 2-3 vehicle challengers, too few
                      # to support either conclusion. ~20 s each, so ~8 min.
DE_KWARGS = dict(popsize=10, maxiter=60, tol=1e-2, seed=1)

PG_USER = "avi"
PG_PASSWORD = "avi"   # a real (non-empty) password — see cell 4 for why '' cannot work
PG_DB = "av_stress"

In [ ]:
import os, subprocess, sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present — pulling latest main")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)
else:
    print(f"Cloning {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0] =", sys.path[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(SHARD_PATH), (
    f"Shard not found at {SHARD_PATH}. Check the Drive mount and the path before "
    f"running anything else — every later cell depends on this file."
)
size_gb = os.path.getsize(SHARD_PATH) / (1024 ** 3)
print(f"Shard found: {SHARD_PATH}")
print(f"Size: {size_gb:.2f} GB")

## 4. Postgres + PostGIS

Colab has no Postgres preinstalled. This installs it fresh, creates role `avi` and database
`av_stress`, enables PostGIS, and runs both schema-init functions. **Colab's disk resets
between sessions, so this database is per-session scratch — that is fine for a validation
run**; nothing here needs to persist.

`PGUSER`/`PGDATABASE`/`PGPASSWORD` are exported into the environment now because
`src/api/config.py` builds its `Settings()` singleton **at import time** — cell 14 must not
be the first thing to read these.

**The role needs a real password, not `''`.** An empty string sets Postgres's password
field to NULL (Postgres itself warns and clears it), which cannot satisfy password
authentication — and TCP connections to `localhost` use `scram-sha-256` by default on
Ubuntu, so `db.get_connection()` would fail immediately with
`fe_sendauth: no password supplied`. `PG_PASSWORD` from the config cell is used both when
creating the role and when exporting `PGPASSWORD`, so the two can never drift apart.

**The PostGIS package name is discovered at runtime, not hardcoded.** Colab's base Ubuntu
image drifts between sessions, and a hardcoded `postgresql-14-postgis-3` breaks outright the
day the image ships Postgres 15 or 16. Installing plain `postgresql` first and reading
`/usr/lib/postgresql/` back tells us which major version actually landed, so the PostGIS
package name is always right for the image running right now. If the specific
`-postgis-3` package genuinely doesn't exist for that major version, this fails loudly with
the actual apt-cache search output rather than raising anywhere from a wrong package name.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib > /dev/null

# Discover the Postgres major version actually installed, rather than hardcoding one —
# Colab's base image drifts between sessions and a fixed version number breaks silently.
import os as _os
pg_versions = sorted(_os.listdir('/usr/lib/postgresql'))
assert pg_versions, "postgresql install did not create /usr/lib/postgresql — apt failed silently"
pg_major = pg_versions[-1]
postgis_pkg = f"postgresql-{pg_major}-postgis-3"
print(f"Detected Postgres major version: {pg_major}  ->  installing {postgis_pkg}")

import subprocess as sp
result = sp.run(["apt-get", "install", "-y", "-qq", postgis_pkg], capture_output=True, text=True)
if result.returncode != 0:
    print(f"FAILED to install {postgis_pkg}. apt-cache search results for 'postgis':")
    search = sp.run(["apt-cache", "search", "postgis"], capture_output=True, text=True)
    print(search.stdout)
    raise RuntimeError(
        f"{postgis_pkg} is not installable on this image. See the candidates printed "
        f"above and adjust postgis_pkg by hand for this session."
    )

!service postgresql start

sp.run(["sudo", "-u", "postgres", "psql", "-c",
        f"CREATE ROLE {PG_USER} WITH SUPERUSER LOGIN PASSWORD '{PG_PASSWORD}';"],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "createdb", "-O", PG_USER, PG_DB],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "psql", "-d", PG_DB, "-c",
        "CREATE EXTENSION IF NOT EXISTS postgis;"], check=True)

os.environ['PGHOST'] = 'localhost'
os.environ['PGPORT'] = '5432'
os.environ['PGUSER'] = PG_USER
os.environ['PGPASSWORD'] = PG_PASSWORD
os.environ['PGDATABASE'] = PG_DB

print(f"Postgres running, role={PG_USER}, database={PG_DB}")

In [ ]:
from src.scoring import db
from src.scoring.export_geometry import init_geometry_schema

conn = db.get_connection()
db.init_schema(conn)
init_geometry_schema(conn)

with conn.cursor() as cur:
    cur.execute("SELECT PostGIS_Version();")
    print("PostGIS_Version():", cur.fetchone()[0])
conn.commit()
print("Schema initialized: scenario_scores, scenario_agents, perturbed_paths")

## 5. Smoke test — one scenario

The cheapest possible check that Phase 1 still works against this specific shard, before
any batch work. If this cell fails, nothing downstream will work either.

In [ ]:
from src.data.loader import ShardLoader
from src.data.parser import ScenarioParser

loader = ShardLoader(SHARD_PATH)
raw = next(iter(loader))
parser = ScenarioParser(raw)

states = parser.get_agent_states()
validity = parser.get_agent_validity()
types = parser.get_agent_types()
sdc_idx = parser.get_sdc_index()

print("scenario_id:", parser.get_scenario_id())
print("states.shape:", states.shape)
print("validity.shape:", validity.shape)
print("sdc_index:", sdc_idx)

import numpy as np
uniq, counts = np.unique(types, return_counts=True)
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
print("agent type counts:")
for u, c in zip(uniq, counts):
    print(f"  {type_names.get(int(u), f'unknown({u})'):10s} {c}")

## 6. Timing probe, then Pass 1 (`score_shard`)

PET rebuilds swept-polygon unions inside `compute_pet_pair`, so its cost grows with agent
density — something the synthetic fixtures (2 agents) never exercised. This probes 3 real
scenarios first and **projects the cost for `MAX_SCENARIOS`**, so the run can be resized
before committing to it rather than discovered 20 minutes in.

In [ ]:
import time
from src.scoring.batch_scorer import _score_one

probe_records = []
t0 = time.time()
for i, raw in enumerate(ShardLoader(SHARD_PATH)):
    if i >= 3:
        break
    p = ScenarioParser(raw)
    rec = _score_one(p.get_agent_states(), p.get_agent_validity(),
                     p.get_scenario_id(), p.get_sdc_index())
    probe_records.append(rec)
probe_elapsed = time.time() - t0
per_scenario = probe_elapsed / len(probe_records)

print(f"Probe: {len(probe_records)} scenarios in {probe_elapsed:.2f}s "
      f"({per_scenario:.2f}s/scenario)")
print(f"Projected for MAX_SCENARIOS={MAX_SCENARIOS}: "
      f"{per_scenario * MAX_SCENARIOS:.1f}s (~{per_scenario * MAX_SCENARIOS / 60:.1f} min)")
print(f"Projected for a full 1000-shard dataset at this rate: "
      f"~{per_scenario * MAX_SCENARIOS * 1000 / 3600:.1f} hours "
      f"(scaling this probe's per-scenario cost, not a measured full run)")

In [ ]:
from src.scoring.batch_scorer import score_shard

t0 = time.time()
records, errors = score_shard(SHARD_PATH, max_scenarios=MAX_SCENARIOS,
                              pet_max_pairs=50, progress_every=25, verbose=True)
pass1_elapsed = time.time() - t0
print(f"\nPass 1 done: {len(records)} scored, {len(errors)} errored, "
      f"{pass1_elapsed:.1f}s total")

## 7. Pass 1 diagnostics

The most valuable cell in the batch section. Covers: error isolation (the first real
exercise of the per-scenario try/except in `score_shard`), the fragility distribution, TTC/PET
saturation rates, agent-count distribution, and validity-gap statistics across every parsed
track — including **interior gaps**, the case that breaks any "vertex index equals timestep"
assumption (see cell 13).

In [ ]:
import numpy as np
from collections import Counter

# ── error isolation ──────────────────────────────────────────────────────────
print("=" * 70)
print("ERROR ISOLATION")
print("=" * 70)
print(f"scored: {len(records)}   errored: {len(errors)}")
if errors:
    exc_types = Counter(e['error'].split(':')[0] for e in errors)
    print("distinct exception types:")
    for exc, n in exc_types.most_common():
        print(f"  {exc:30s} {n}")
else:
    print("0 errors — clean pass on this sample")

# ── fragility distribution ───────────────────────────────────────────────────
frag = np.array([r['fragility_score'] for r in records])
print()
print("=" * 70)
print("FRAGILITY SCORE DISTRIBUTION")
print("=" * 70)
print(f"min={frag.min():.4f}  max={frag.max():.4f}  mean={frag.mean():.4f}")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:<3d} {np.percentile(frag, p):.4f}")

# ── TTC / PET saturation — SDC-restricted (production) vs all-pairs ────────────
# Import the REAL constants rather than hardcoding literals — a diagnostic that
# reimplements a threshold with a literal will silently drift from the function
# it is describing. (An earlier draft of this notebook had exactly that bug: it
# compared `== 0.0` when the real clamp is `<= TTC_FLOOR`, and `== 999.0` when the
# real sentinel test is `< TTC_INFINITY`. Both under-counted saturation.)
from src.danger.danger_score import TTC_FLOOR, PET_FLOOR
from src.danger.ttc_engine import TTC_INFINITY
from src.danger.pet_engine import PET_INFINITY

# After the Phase 3 rework, score_scenario returns BOTH: min_ttc/min_pet are the
# SDC-restricted values it now ranks on, and min_ttc_all_pairs/min_pet_all_pairs
# are the old scene-wide numbers, kept as a diagnostic. The direct before/after
# for finding 1 is comparing the saturation of the two, on the same scenarios.
ttc_sdc = np.array([r['min_ttc'] for r in records])
pet_sdc = np.array([r['min_pet'] for r in records])
ttc_all = np.array([r['min_ttc_all_pairs'] for r in records])
pet_all = np.array([r['min_pet_all_pairs'] for r in records])

def _sat(arr, floor):
    # (exact-zero mask, floor-capped mask, sentinel mask). Floor-capped is what
    # actually pins the danger signal at its max: compute_danger_score clamps via
    # max(x, floor), so anything <= floor behaves identically to an exact 0.0.
    is_ttc = (floor == TTC_FLOOR)
    sentinel = arr >= (TTC_INFINITY if is_ttc else PET_INFINITY)
    return (arr == 0.0), (arr <= floor), sentinel

ttc_sdc_zero, ttc_sdc_cap, ttc_sdc_sent = _sat(ttc_sdc, TTC_FLOOR)
ttc_all_zero, ttc_all_cap, ttc_all_sent = _sat(ttc_all, TTC_FLOOR)
pet_sdc_zero, pet_sdc_cap, pet_sdc_sent = _sat(pet_sdc, PET_FLOOR)
pet_all_zero, pet_all_cap, pet_all_sent = _sat(pet_all, PET_FLOOR)

pet_sdc_negative = (pet_sdc < 0.0)   # finding 5, production signal, full sample

print()
print("=" * 70)
print("TTC / PET SATURATION (finding 1) — SDC-restricted vs all-pairs")
print("=" * 70)
print(f"                              SDC-restricted (ranked on)   all-pairs (old)")
print(f"min_ttc <= TTC_FLOOR ({TTC_FLOOR}):   "
      f"{ttc_sdc_cap.sum():4d}/{len(ttc_sdc)} ({100*ttc_sdc_cap.mean():5.1f}%)        "
      f"{ttc_all_cap.sum():4d}/{len(ttc_all)} ({100*ttc_all_cap.mean():5.1f}%)")
print(f"min_pet <= PET_FLOOR ({PET_FLOOR}):   "
      f"{pet_sdc_cap.sum():4d}/{len(pet_sdc)} ({100*pet_sdc_cap.mean():5.1f}%)        "
      f"{pet_all_cap.sum():4d}/{len(pet_all)} ({100*pet_all_cap.mean():5.1f}%)")
print(f"min_ttc >= INFINITY sentinel:  "
      f"{ttc_sdc_sent.sum():4d}/{len(ttc_sdc)} ({100*ttc_sdc_sent.mean():5.1f}%)        "
      f"{ttc_all_sent.sum():4d}/{len(ttc_all)} ({100*ttc_all_sent.mean():5.1f}%)")
print()
if ttc_all_cap.mean() > 0.5 and ttc_sdc_cap.mean() < ttc_all_cap.mean() - 0.1:
    print(f"-> FINDING 1 CONFIRMED FIXED: all-pairs TTC saturated "
          f"{100*ttc_all_cap.mean():.0f}% of scenarios (uninformative); "
          f"SDC-restricted saturates {100*ttc_sdc_cap.mean():.0f}%, leaving "
          f"{(~ttc_sdc_cap).sum()} scenarios with a discriminating TTC signal.")
elif ttc_all_cap.mean() <= 0.5:
    print(f"-> all-pairs TTC was NOT heavily saturated on this sample "
          f"({100*ttc_all_cap.mean():.0f}%) — smaller shard slice than the "
          f"100/100 validation run, or a quieter shard region.")
else:
    print(f"-> SDC-restricted saturation ({100*ttc_sdc_cap.mean():.0f}%) is not "
          f"materially below all-pairs ({100*ttc_all_cap.mean():.0f}%). Investigate "
          f"before trusting the new ranking on this data.")
print()
print(f"min_pet < 0.0, SDC-restricted (finding 5 — now genuine overlaps only): "
      f"{pet_sdc_negative.sum()}/{len(pet_sdc)} ({100*pet_sdc_negative.mean():.1f}%)")

# non-saturated fragility percentiles: exclude scenarios whose PRODUCTION signal
# (SDC-restricted TTC) is floor-capped. frag is already the SDC-restricted score.
non_sat = frag[~ttc_sdc_cap]
print()
print(f"fragility percentiles over the NON-saturated subset (SDC-restricted TTC "
      f"above floor, n={len(non_sat)}):")
if len(non_sat):
    for p in [10, 25, 50, 75, 90, 95, 99]:
        print(f"  p{p:<3d} {np.percentile(non_sat, p):.4f}")
else:
    print("  (every scenario's SDC-restricted TTC saturated — unexpected post-rework, "
          "see the summary cell)")

# names cell 48 still references
ttc_floor_capped = ttc_sdc_cap
pet_negative = pet_sdc_negative

# ── agent count distribution ──────────────────────────────────────────────────
n_agents = np.array([r['n_agents'] for r in records])
print()
print("=" * 70)
print("AGENT COUNT (n_agents) DISTRIBUTION")
print("=" * 70)
print(f"min={n_agents.min()}  median={int(np.median(n_agents))}  max={n_agents.max()}")

# ── PET pair-selection diagnostics (finding 4) — index arithmetic only, no geometry ──
print()
print("=" * 70)
print("PET PAIR-SELECTION DIAGNOSTICS (finding 4)")
print("=" * 70)
print("NOTE post-rework: the production ranking now uses compute_min_pet_sdc, which")
print("is SDC-anchored and UNCAPPED, so this degeneracy no longer affects the ranking.")
print("It still describes min_pet_all_pairs, the retained scene-density diagnostic.")


def _pet_pairs_checked(n_agents_i, max_pairs=50):
    """Replicate compute_min_pet_scenario's (i, j) selection with i<j, no geometry."""
    pairs = []
    for i in range(n_agents_i):
        for j in range(i + 1, n_agents_i):
            if len(pairs) >= max_pairs:
                return pairs
            pairs.append((i, j))
    return pairs


ge_51 = 0
sdc_never_checked = 0
anchor_counts = []
for r in records:
    n = r['n_agents']
    if n >= 51:
        ge_51 += 1
    pairs = _pet_pairs_checked(n)
    anchors = len(set(i for i, j in pairs))
    anchor_counts.append(anchors)
    # sdc_idx is not stored on the record; re-derive is not available here without
    # re-parsing. This diagnostic instead reports the STRUCTURAL degeneracy (anchor
    # count) directly — see cell 7d for the SDC-specific version, computed on the
    # cached sample where sdc_idx is available.

print(f"scenarios with n_agents >= 51 (cap exhausted inside i=0): "
      f"{ge_51}/{len(records)} ({100*ge_51/len(records):.1f}%)")
print(f"distinct 'i' anchors reached per scenario: "
      f"min={min(anchor_counts)}  median={int(np.median(anchor_counts))}  "
      f"max={max(anchor_counts)}")
print("(anchors == 1 means every checked PET pair shares agent 0 in that scenario;")
print(" the SDC-specific version of this — whether the SDC is ever checked at all —")
print(" is computed in cell 7d against the cached sample, where sdc_idx is available.)")

# ── validity-gap statistics ───────────────────────────────────────────────────
print()
print("=" * 70)
print("VALIDITY-GAP STATISTICS")
print("=" * 70)
print("(recomputed per scenario in this pass — see cell 7b for the cached-sample version")
print(" used by cells 7c/7d; this section re-walks the shard once more, which is the")
print(" price of getting gap stats on the FULL MAX_SCENARIOS sample rather than DIAG_N)")

fully_valid = prefix_suffix_only = interior_gap = total_tracks = 0
t0 = time.time()
wanted_ids = set(r['scenario_id'] for r in records)
for raw in ShardLoader(SHARD_PATH):
    if not wanted_ids:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in wanted_ids:
        continue
    wanted_ids.discard(sid)
    v = p.get_agent_validity()
    for i in range(v.shape[0]):
        total_tracks += 1
        row = v[i]
        if row.all():
            fully_valid += 1
            continue
        valid_idx = np.where(row)[0]
        if len(valid_idx) == 0:
            continue
        # contiguous prefix/suffix only: the valid indices form one unbroken run
        span = valid_idx[-1] - valid_idx[0] + 1
        if span == len(valid_idx):
            prefix_suffix_only += 1
        else:
            interior_gap += 1
gap_scan_elapsed = time.time() - t0

print(f"total agent tracks examined: {total_tracks}  ({gap_scan_elapsed:.1f}s)")
print(f"  fully valid (all 91 timesteps):        {fully_valid} "
      f"({100*fully_valid/max(total_tracks,1):.1f}%)")
print(f"  contiguous run (no interior gap):       {prefix_suffix_only} "
      f"({100*prefix_suffix_only/max(total_tracks,1):.1f}%)")
print(f"  INTERIOR GAP (breaks index==timestep):   {interior_gap} "
      f"({100*interior_gap/max(total_tracks,1):.1f}%)")

# ── timing projection ──────────────────────────────────────────────────────
print()
print("=" * 70)
print("WALL-CLOCK PROJECTION")
print("=" * 70)
mean_sec = np.mean([r['score_seconds'] for r in records])
print(f"mean seconds/scenario (this pass): {mean_sec:.3f}s")
print(f"projected for a full 1000-shard dataset at ~{mean_sec:.3f}s/scenario "
      f"and this shard's scenario count as a stand-in: see the cell-6 probe for the "
      f"MAX_SCENARIOS-scale projection; this line uses the Pass-1-measured rate instead "
      f"of the 3-scenario probe rate, so compare the two for probe reliability.")

## 7b. Diagnostic sample cache

Cells 7c and 7d both need parsed arrays (`states`, `validity`, `types`, `sdc_idx`) for the
same sample of scenarios, and the shard lives on Drive-mounted storage where a second and
third sequential read would be the dominant cost of running this section. So this cell
parses `DIAG_N` scenarios once and holds them in memory.

Cheap to hold: a 100-agent scenario is `91 × 100 × 7 × 4 bytes ≈ 255 KB`, so `DIAG_N=50`
scenarios is on the order of tens of MB — printed below rather than assumed.

In [ ]:
import sys as _sys

diag_cache = []
for raw in ShardLoader(SHARD_PATH):
    if len(diag_cache) >= DIAG_N:
        break
    p = ScenarioParser(raw)
    diag_cache.append({
        'scenario_id': p.get_scenario_id(),
        'states': p.get_agent_states(),
        'validity': p.get_agent_validity(),
        'types': p.get_agent_types(),
        'sdc_idx': p.get_sdc_index(),
    })

resident_bytes = sum(
    d['states'].nbytes + d['validity'].nbytes + d['types'].nbytes
    for d in diag_cache
)
print(f"Cached {len(diag_cache)} scenarios for cells 7c/7d "
      f"({resident_bytes / 1024**2:.1f} MB resident)")

## 7c. Rank-correlation experiment — confirms the finding-1 fix

Computes each cached scenario's danger profile **twice**, changing only the pair set, and
both paths are now real production code (before the rework, the SDC-only path was
hand-reimplemented here — that reimplementation-drift risk is why the old 7d needed a
validity gate):

- **all-pairs (old ranking)** — `compute_danger_score(compute_min_ttc_scenario(...),
  compute_min_pet_scenario(...))`, the scene-wide functions Phase 3 used to rank on
- **SDC-restricted (new ranking)** — `score_scenario(...)['fragility_score']`, exactly what
  Pass 1 now produces

Both go through `rank_scenarios`. This is the concrete before/after: which scenarios the
old ranking sent to Phase 4 that the new one does not, and vice versa.

**Interpretation:** a low Spearman rho with small top-N overlap is the fix working — the two
signals genuinely rank scenarios differently, and the validation run measured rho = -0.0152
with 2/5 overlap on this data. A rho near 1.0 would mean the rework changed nothing and the
motivation was wrong.

Cost note: `score_scenario` runs BOTH pair sets internally (SDC-restricted for the score,
all-pairs for the diagnostic), so the all-pairs recompute below is redundant work — kept
explicit so the comparison reads clearly. `DIAG_N` is tuned from the cell-6 probe.

In [ ]:
from src.danger.ttc_engine import compute_min_ttc_scenario
from src.danger.pet_engine import compute_min_pet_scenario
from src.danger.danger_score import compute_danger_score, score_scenario
from src.scoring.ranker import rank_scenarios

t0 = time.time()
all_pairs_records = []
sdc_prod_records = []

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']

    # ---- old ranking: scene-wide all-pairs ----
    min_ttc_all = compute_min_ttc_scenario(states, validity)
    min_pet_all = compute_min_pet_scenario(states, validity, max_pairs=50)
    frag_all = compute_danger_score(min_ttc_all, min_pet_all)
    all_pairs_records.append({'scenario_id': d['scenario_id'], 'fragility_score': frag_all,
                              'min_ttc': min_ttc_all, 'min_pet': min_pet_all})

    # ---- new ranking: exactly what Pass 1 now writes ----
    rec = score_scenario(states, validity, d['scenario_id'], sdc_idx)
    sdc_prod_records.append({'scenario_id': d['scenario_id'],
                             'fragility_score': rec['fragility_score'],
                             'min_ttc': rec['min_ttc'], 'min_pet': rec['min_pet']})

elapsed_7c = time.time() - t0
print(f"7c compute: {elapsed_7c:.1f}s for {len(diag_cache)} scenarios "
      f"({elapsed_7c/len(diag_cache):.2f}s/scenario)")

In [ ]:
# AUDIT A14. Named for its experiment rather than a bare `rho`. Two cells computed
# a Spearman correlation into the same global and the summary read whichever ran
# last under THIS experiment's label. An AST sweep over every code cell found
# exactly one such shadowed read in the whole notebook, and this was it.
from scipy.stats import spearmanr

ranked_all = rank_scenarios(all_pairs_records)
ranked_sdc = rank_scenarios(sdc_prod_records)

order_all = [r['scenario_id'] for r in ranked_all]
order_sdc = [r['scenario_id'] for r in ranked_sdc]

rank_all = {sid: i for i, sid in enumerate(order_all)}
rank_sdc = {sid: i for i, sid in enumerate(order_sdc)}
common = list(rank_all.keys())

rho_all_pairs_vs_sdc, pval_all_pairs_vs_sdc = spearmanr([rank_all[s] for s in common], [rank_sdc[s] for s in common])

top10_all = set(order_all[:10])
top10_sdc = set(order_sdc[:10])
topN_all = set(order_all[:TOP_N])
topN_sdc = set(order_sdc[:TOP_N])

frag_all_by = {r['scenario_id']: r['fragility_score'] for r in ranked_all}
frag_sdc_by = {r['scenario_id']: r['fragility_score'] for r in ranked_sdc}

print("=" * 70)
print("RANK-CORRELATION: old all-pairs ranking vs new SDC-restricted ranking")
print("=" * 70)
print(f"Spearman rho: {rho_all_pairs_vs_sdc:.4f}  (p={pval_all_pairs_vs_sdc:.4g})   [validation run measured -0.0152]")
print(f"top-10 set overlap: {len(top10_all & top10_sdc)}/10")
print(f"top-{TOP_N} set overlap (the set Phase 4 actually runs on): "
      f"{len(topN_all & topN_sdc)}/{TOP_N}")

print()
print(f"scenarios in the OLD top-{TOP_N} that the NEW ranking drops:")
for sid in topN_all - topN_sdc:
    print(f"  {sid}  old_frag={frag_all_by[sid]:.3f} (rank {rank_all[sid]})  ->  "
          f"new_frag={frag_sdc_by[sid]:.4f} (rank {rank_sdc[sid]})")
print(f"scenarios the NEW top-{TOP_N} adds:")
for sid in topN_sdc - topN_all:
    print(f"  {sid}  new_frag={frag_sdc_by[sid]:.3f} (rank {rank_sdc[sid]})  <-  "
          f"old_frag={frag_all_by[sid]:.4f} (rank {rank_all[sid]})")

print()
if abs(rho_all_pairs_vs_sdc) < 0.3 and len(topN_all & topN_sdc) < TOP_N:
    print("VERDICT: the two rankings are effectively unrelated — the rework materially")
    print("changed which scenarios reach Phase 4, consistent with the validation run.")
else:
    print(f"VERDICT: rho={rho_all_pairs_vs_sdc:.3f}, top-{TOP_N} overlap {len(topN_all & topN_sdc)}/{TOP_N}.")
    print("Larger overlap than the validation run — check whether this shard slice is")
    print("less crowded, so all-pairs was not saturating and the two signals agree.")

## 7d. PET sign-semantics regression — confirms the finding-5 fix, now visit-aware

`compute_pet_pair` evaluates `max(enter_b - exit_a, enter_a - exit_b)` over **every pairing
of one A-visit with one B-visit**, so a negative result can only mean that some single
visit of A genuinely overlaps some single visit of B.

**This cell had the bug it was checking for.** Until Batch 4 it verified a negative PET by
recomputing *merged* occupancy spans with `_occupancy_interval` and asserting
`ea <= xb and eb <= xa`. Fed the audit's own B06 scene — A present at frames 0,1,5,6 and B
only at frame 3, sharing no frame at all — that check returns `True` and reports **no
regression**. It would have certified the exact defect B06 fixes. The verification now
recomputes visit lists with `_occupancy_visits` and requires that *some specific pair of
visits* overlaps, which is the property the function actually promises.

Counters keep their old names and format so the negative-pair rate stays directly
comparable to the v3 validation run (56% negative, a third of them false alarms).
`n_multi_visit_pairs` and `n_multi_visit_decisive` are new: the first says whether B06's
condition ever arises on real data, the second whether it ever **changed the answer** —
i.e. the winning visit pair was not simply the first one. A large first number with a zero
second would mean this fix was theoretical-correctness only.

No reimplementation, so no validity gate: everything is imported from the module under test.

In [ ]:
from src.danger.pet_engine import (
    compute_pet_pair, get_path_polygon, _occupancy_visits, PET_INFINITY, DT,
)

n_sdc_pairs = 0
n_negative = 0
n_zero = 0
n_positive = 0
n_infinity = 0
n_multi_visit_pairs = 0       # at least one agent visited the zone more than once
n_multi_visit_later_pair = 0  # ...and the minimum came from a pair other than the first
n_multi_visit_changed = 0     # ...and the MERGED-SPAN answer differs (the real measure)
n_multi_visit_signflip = 0    # ...and it differs in SIGN, which changes the conclusion
worst_change = 0.0            # largest |merged - corrected| seen, in seconds
regressions = []              # negative PET with no overlapping VISIT pair — must stay empty

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']
    N = states.shape[0]
    sdc_path = get_path_polygon(states, sdc_idx, validity)
    if sdc_path is None:
        continue
    for j in range(N):
        if j == sdc_idx:
            continue
        pet = compute_pet_pair(states, validity, sdc_idx, j, path_a=sdc_path)
        n_sdc_pairs += 1
        if pet >= PET_INFINITY:
            n_infinity += 1
            continue

        other_path = get_path_polygon(states, j, validity)
        if other_path is None:
            continue
        zone = sdc_path.intersection(other_path)
        visits_a = _occupancy_visits(states, validity, sdc_idx, zone)
        visits_b = _occupancy_visits(states, validity, j, zone)

        # AUDIT A13. Did B06's condition arise, and did it CHANGE THE ANSWER?
        #
        # The old measure here counted "the minimum came from a pair other than the
        # first", which is a proxy for the question and not the question. On B06's own
        # canonical fixture it gets the answer exactly backwards: with
        # visits_a=[(0,1),(5,6)] and visits_b=[(3,3)] both pairings score +0.2, so the
        # first pair is AMONG the winners and the proxy reports zero — while the
        # pre-B06 merged-span answer for that same fixture is -0.3. A half-second
        # change that flips sign, reported as "never decided anything".
        #
        # So the before/after measure now computes the OLD answer explicitly, on the
        # same visits, and compares it directly. The first-pair counter is KEPT because
        # "how often does the minimum come from a later pair" is a real and separate
        # fact — but it is no longer the before/after measure, and the two are printed
        # apart so they cannot be read as the same number again.
        if len(visits_a) > 1 or len(visits_b) > 1:
            n_multi_visit_pairs += 1
            scored = [(max(eb - xa, ea - xb) * DT, i_a, i_b)
                      for i_a, (ea, xa) in enumerate(visits_a)
                      for i_b, (eb, xb) in enumerate(visits_b)]
            best_value = min(v for v, _, _ in scored)
            winners = [(i_a, i_b) for v, i_a, i_b in scored if v == best_value]
            if (0, 0) not in winners:
                n_multi_visit_later_pair += 1

            # The pre-B06 implementation merged each agent's visits into ONE span:
            # entry = first visit's entry, exit = last visit's exit.
            merged_value = max(visits_b[0][0] - visits_a[-1][1],
                               visits_a[0][0] - visits_b[-1][1]) * DT
            if merged_value != best_value:
                n_multi_visit_changed += 1
                worst_change = max(worst_change, abs(merged_value - best_value))
                if (merged_value < 0) != (best_value < 0):
                    n_multi_visit_signflip += 1
                print(f"    [B06 changed] {d['scenario_id']} pair=({sdc_idx},{j}): "
                      f"merged-span {merged_value:+.4f} s -> corrected "
                      f"{best_value:+.4f} s")

        if pet > 0:
            n_positive += 1
            continue
        if pet == 0:
            n_zero += 1
            continue

        # pet < 0 — require that some SPECIFIC visit pair genuinely overlaps
        n_negative += 1
        overlaps = any(ea <= xb and eb <= xa
                       for ea, xa in visits_a
                       for eb, xb in visits_b)
        if not overlaps:
            regressions.append((d['scenario_id'], sdc_idx, j, pet, visits_a, visits_b))

print("=" * 70)
print("PET SIGN-SEMANTICS REGRESSION (finding 5) + VISIT SEPARATION (audit B06)")
print("=" * 70)
print(f"SDC pairs evaluated:            {n_sdc_pairs}")
print(f"  PET_INFINITY (no shared zone / no crossing): {n_infinity}")
print(f"  positive (sequenced crossing, safe):         {n_positive}")
print(f"  exactly zero:                                {n_zero}")
print(f"  negative (genuine simultaneous occupancy):   {n_negative}")
if n_sdc_pairs:
    print(f"  negative rate: {100.0 * n_negative / n_sdc_pairs:.1f}%"
          f"   [v3 validation run measured 56%]")
print()
print("audit B06 — did separate visits actually occur on real data?")
print(f"  pairs where an agent visited the zone more than once: {n_multi_visit_pairs}")
print()
print("  THE BEFORE/AFTER MEASURE (merged-span answer vs the corrected engine, same")
print("  inputs, computed directly rather than inferred from which pair won):")
print(f"    pairs where the answer CHANGED:        {n_multi_visit_changed}")
print(f"    ...where it changed SIGN:              {n_multi_visit_signflip}")
print(f"    largest change seen:                   {worst_change:.4f} s")
print()
print("  A SEPARATE STATISTIC, not the before/after measure (audit A13 — these used to")
print("  be one number, and it answered the wrong question):")
print(f"    minimum came from a pair other than the first: {n_multi_visit_later_pair}")
if n_multi_visit_pairs == 0:
    print("  -> B06 never fires on this sample; the fix is correctness-only here.")
elif n_multi_visit_changed == 0:
    print("  -> multi-visit occurs but the merged answer agreed on every pair here.")
else:
    print("  -> B06 changed real PET values; pre-Batch-4 rankings used the merged answer.")

print()
if regressions:
    print(f"*** REGRESSION: {len(regressions)} negative PET(s) with NO overlapping visit pair.")
    print("*** compute_pet_pair returned a negative value that is not a genuine overlap.")
    for sid, a, b, pet, va, vb in regressions[:10]:
        print(f"    {sid} pair=({a},{b}) pet={pet:.2f} a_visits={va} b_visits={vb}")
    raise AssertionError(f"{len(regressions)} PET sign-semantics regressions — see above")
else:
    print("OK — every negative PET corresponds to a genuinely overlapping pair of visits.")
    print("Finding 5's semantics and audit B06's visit separation both hold on real data.")

## 7e. TTC discrimination rate — measures the audit B07 fix

Block 3 v3 recorded that the all-pairs minimum TTC saturated to 0.0 on **100/100** scenarios,
and that restricting to SDC pairs brought that to **70/100**. Both numbers were measured
before audit B07 was fixed, when TTC was the linear projection
`(dist - safe_dist) / closing_speed`.

That formula reports a finite TTC for any pair that is *currently closing in the radial
sense*, including pairs whose paths never bring them within a safe distance — the audit's
repro returned 1.76 s for two circles that miss by 1 m. The quadratic root returns
`TTC_INFINITY` for those, so the distribution of TTC values across a shard should shift
toward infinity. This cell measures where it actually lands, in the same `n/N` form.

The breakdown by **reason** is the part that matters for future work: an infinity from
`D < 0` means the paths genuinely never intersect, which is B07 doing its job; an infinity
from `closing_speed <= 0` means the agents are simply diverging, which the old code already
got right. Only the first number is attributable to this batch. That split is also the input
the deferred TTC saturation work needs — it cannot be designed against a pre-B07 rate.

In [ ]:
import numpy as np
from src.danger.ttc_engine import (
    TTC_INFINITY, compute_effective_radius, compute_min_ttc_sdc,
    compute_min_ttc_scenario,
)
from src.danger.danger_score import TTC_FLOOR

n_scen = 0
n_sat_all = 0        # all-pairs min TTC at the floor  [v3 measured 100/100]
n_sat_sdc = 0        # SDC-restricted at the floor     [v3 measured  70/100]
n_finite_sdc = 0
n_inf_sdc = 0

# why each SDC pair-timestep that produced no finite TTC produced none
reason_miss = 0      # discriminant < 0: paths never bring the circles together
reason_diverging = 0 # b >= 0: already separating, old code was right too
reason_overlap = 0   # c <= 0: already overlapping, returns 0.0

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']
    n_scen += 1
    if compute_min_ttc_scenario(states, validity) <= TTC_FLOOR:
        n_sat_all += 1
    m = compute_min_ttc_sdc(states, validity, sdc_idx)
    if m <= TTC_FLOOR:
        n_sat_sdc += 1
    if m >= TTC_INFINITY:
        n_inf_sdc += 1
    else:
        n_finite_sdc += 1

    # attribute the SDC pairs, timestep by timestep, to a branch
    N, T = states.shape[0], states.shape[1]
    radii = compute_effective_radius(states[:, :, 5].astype(np.float64),
                                     states[:, :, 6].astype(np.float64))
    for t in range(T):
        if not validity[sdc_idx, t]:
            continue
        for j in range(N):
            if j == sdc_idx or not validity[j, t]:
                continue
            dx = float(states[j, t, 0]) - float(states[sdc_idx, t, 0])
            dy = float(states[j, t, 1]) - float(states[sdc_idx, t, 1])
            dvx = float(states[j, t, 2]) - float(states[sdc_idx, t, 2])
            dvy = float(states[j, t, 3]) - float(states[sdc_idx, t, 3])
            R = float(radii[sdc_idx, t]) + float(radii[j, t])
            a = dvx * dvx + dvy * dvy
            b = 2.0 * (dx * dvx + dy * dvy)
            c = dx * dx + dy * dy - R * R
            if c <= 0:
                reason_overlap += 1
            elif a == 0 or b >= 0:
                reason_diverging += 1
            elif b * b - 4.0 * a * c < 0:
                reason_miss += 1

print("=" * 70)
print("TTC DISCRIMINATION RATE (audit B07)")
print("=" * 70)
print(f"scenarios sampled: {n_scen}")
print(f"  all-pairs min TTC at the floor:      {n_sat_all}/{n_scen}"
      f"   [v3 measured 100/100]")
print(f"  SDC-restricted min TTC at the floor: {n_sat_sdc}/{n_scen}"
      f"   [v3 measured  70/100]")
print(f"  SDC-restricted finite (a real TTC):  {n_finite_sdc}/{n_scen}")
print(f"  SDC-restricted TTC_INFINITY:         {n_inf_sdc}/{n_scen}")
print()
print("SDC pair-timesteps with no finite TTC, by reason:")
print(f"  already overlapping (c <= 0, returns 0.0):        {reason_overlap}")
print(f"  diverging (old code also said infinity):          {reason_diverging}")
print(f"  paths never intersect (D < 0) -- THE B07 FIX:     {reason_miss}")
print()
print("Only the last line is attributable to Batch 4. A large value there means the")
print("pre-B07 ranking was driven substantially by collisions that could not occur.")

## 7f. Before/after on identical inputs — how much did B07 actually move the ranking?

7e measures the post-fix state. This measures the **difference**, which is the only way to
say whether audit B07 was a correctness footnote or a change to which scenarios Phase 4
ever saw.

Both signals are computed on the same cached states: production `compute_min_ttc_sdc`
against a local re-implementation of the **old** linear formula. The old formula is
reproduced here deliberately — it no longer exists in the codebase, and the comparison is
meaningless without it. It is confined to this cell and is not importable.

The Spearman structure mirrors cell 7c so the two experiments read alike. A rho near 1.0
with few changed scenarios would mean B07 was real but inconsequential on this data; a low
rho means the pre-B07 rankings that selected scenarios for stress-testing were selecting on
a signal that did not measure what it claimed to.

In [ ]:
import numpy as np
from scipy.stats import spearmanr
from src.danger.ttc_engine import TTC_INFINITY, compute_effective_radius, compute_min_ttc_sdc


def _old_linear_min_ttc_sdc(states, validity, sdc_index):
    """The pre-Batch-4 formula, reproduced ONLY for this comparison."""
    N, T = states.shape[0], states.shape[1]
    if not (0 <= sdc_index < N):
        return TTC_INFINITY
    radii = compute_effective_radius(states[:, :, 5].astype(np.float64),
                                     states[:, :, 6].astype(np.float64))
    best = TTC_INFINITY
    for t in range(T):
        if not validity[sdc_index, t]:
            continue
        for j in range(N):
            if j == sdc_index or not validity[j, t]:
                continue
            dx = float(states[j, t, 0]) - float(states[sdc_index, t, 0])
            dy = float(states[j, t, 1]) - float(states[sdc_index, t, 1])
            dist = np.sqrt(dx * dx + dy * dy)
            safe = float(radii[sdc_index, t]) + float(radii[j, t])
            if dist <= safe:
                return 0.0
            dvx = float(states[j, t, 2]) - float(states[sdc_index, t, 2])
            dvy = float(states[j, t, 3]) - float(states[sdc_index, t, 3])
            closing = -(dx * dvx + dy * dvy) / dist
            if closing <= 0:
                continue
            best = min(best, (dist - safe) / closing)
    return float(best)


rows = []
for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']
    rows.append({
        'scenario_id': d['scenario_id'],
        'old': _old_linear_min_ttc_sdc(states, validity, sdc_idx),
        'new': compute_min_ttc_sdc(states, validity, sdc_idx),
    })

changed = [r for r in rows if r['old'] != r['new']]
became_inf = [r for r in changed if r['new'] >= TTC_INFINITY > r['old']]
got_larger = [r for r in changed if r['old'] < r['new'] < TTC_INFINITY]
got_smaller = [r for r in changed if r['new'] < r['old']]

order_old = [r['scenario_id'] for r in sorted(rows, key=lambda r: (r['old'], r['scenario_id']))]
order_new = [r['scenario_id'] for r in sorted(rows, key=lambda r: (r['new'], r['scenario_id']))]
rank_old = {s: i for i, s in enumerate(order_old)}
rank_new = {s: i for i, s in enumerate(order_new)}
common = list(rank_old)
rho_ttc_before_after, pval_ttc_before_after = spearmanr([rank_old[s] for s in common], [rank_new[s] for s in common])

print("=" * 70)
print("TTC BEFORE/AFTER (audit B07) — same inputs, two formulas")
print("=" * 70)
print(f"scenarios compared:            {len(rows)}")
print(f"  min_ttc changed:             {len(changed)}")
print(f"    finite -> TTC_INFINITY:    {len(became_inf)}   (the B07 false alarms)")
print(f"    finite -> larger finite:   {len(got_larger)}")
print(f"    decreased:                 {len(got_smaller)}   (expected 0)")
print()
print(f"Spearman rho, old TTC ranking vs new: {rho_ttc_before_after:.4f}  (p={pval_ttc_before_after:.4g})")
print("  [cell 7c measured rho = -0.0152 for the all-pairs vs SDC-restricted change]")
print()
if got_smaller:
    print("*** UNEXPECTED: B07 can only REMOVE collisions that cannot occur, so no")
    print("*** scenario should get a smaller min TTC. Investigate before trusting this run.")
    for r in got_smaller[:10]:
        print(f"    {r['scenario_id']}  old={r['old']:.3f} -> new={r['new']:.3f}")
print("largest changes:")
for r in sorted(changed, key=lambda r: -(min(r['new'], TTC_INFINITY) - r['old']))[:10]:
    print(f"  {r['scenario_id']}  old={r['old']:.3f}  ->  new={r['new']:.3f}")

## 7g. Baseline replay drift — the `max_baseline_drift=None` sweep

Batch 1 built this escape hatch and it has never been used. `PerturbationSpace` takes
`max_baseline_drift=None` specifically so the real distribution can be measured *before*
the 0.5 m default is defended as final, and 0.5 m has been an unexamined guess through five
batches of work built on top of it.

**Full shard, not a sample.** Construction is one `simulate` call, not a DE search —
measured at **3.8 ms per scenario** on a 40-agent/91-frame scene, so ~4 s of compute for a
1000-scenario shard. The real cost is one additional sequential `ShardLoader` pass over
Drive-mounted storage, which cell 7b notes is the dominant cost of this section. This cell
cannot reuse `diag_cache`, which is capped at `DIAG_N`.

**Two gates, reported separately.** The hard gate (`baseline_replay_collides`) fires
regardless of any threshold, so no choice of `max_baseline_drift` changes it; folding it
into the drift distribution would corrupt the number this cell exists to produce.

**The hard gate still raises with `max_baseline_drift=None`** — only the soft branch is
disabled — so those scenarios are recovered from the exception, which carries
`baseline_replay_error` and `baseline_replay_collides` for exactly this purpose.
`has_interior_gap` is *not* on the exception and is recomputed independently, because the
constructor sets it on `self` and then raises, discarding the object.

**Exception ordering is load-bearing:** `ReplayFidelityError` subclasses `ValueError`, so a
bare `except ValueError` placed first would swallow every gate refusal and silently destroy
the measurement.

Percentiles rather than mean/max: a threshold decision needs the shape of the tail. Broken
down by interior validity gap (Batch 1's own prediction) and by challenger agent type —
vehicles use the bicycle model, pedestrians and cyclists the linear one, and Block 2
Concept 6's fidelity argument was derived from vehicle behaviour only.


In [ ]:
import numpy as np
from src.data.validity import has_interior_gap
from src.optimization.perturbation_space import (
    PerturbationSpace, ReplayFidelityError, pick_nearest_challenger,
)

TYPE_NAMES = {0: 'UNSET', 1: 'VEHICLE', 2: 'PEDESTRIAN', 3: 'CYCLIST', 4: 'OTHER'}

rows = []                 # one per scenario that produced a drift measurement
skipped = {'no_challenger': 0, 'unusable_geometry': 0, 'never_valid': 0}
skip_examples = []

for raw in ShardLoader(SHARD_PATH):
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    states, validity = p.get_agent_states(), p.get_agent_validity()
    types, sdc_idx = p.get_agent_types(), p.get_sdc_index()

    tgt = pick_nearest_challenger(states, validity, sdc_idx)
    if tgt < 0:
        skipped['no_challenger'] += 1
        continue

    # Known BEFORE the space is built, so it survives a hard-gate refusal where the
    # object itself is discarded. Agent type matters here for a structural reason:
    # pedestrians and cyclists go through the LINEAR model, which carries no heading
    # state, while Block 2 Concept 6's drift-fidelity argument was derived entirely
    # from vehicle behaviour (turning radius, rear-axle offset). This is the first
    # dataset that can show whether the drift distribution differs by model.
    tgt_type = int(types[tgt])
    gap = bool(has_interior_gap(validity, tgt))

    try:
        space = PerturbationSpace(states, validity, types, sdc_idx, tgt,
                                  max_baseline_drift=None)
        err, collides = space.baseline_replay_error, space.baseline_replay_collides

    # ORDER MATTERS: ReplayFidelityError SUBCLASSES ValueError, so a bare
    # `except ValueError` first would swallow the gate refusals and destroy the whole
    # measurement. This clause must stay above the one below it.
    except ReplayFidelityError as e:
        # With max_baseline_drift=None the soft gate is disabled, so the only refusal
        # reachable here is the hard one. A 'drift' refusal would mean the kwarg is
        # not doing what Batch 1 documents, which would invalidate every number in
        # this cell rather than just one row.
        assert e.reason == 'collision', (
            f"{sid}: got a '{e.reason}' refusal with max_baseline_drift=None — the "
            f"soft gate is still armed and this sweep measures nothing"
        )
        # has_interior_gap was computed above precisely because the constructor sets
        # it on `self` and then raises, discarding the object unrecoverably.
        err, collides = e.baseline_replay_error, e.baseline_replay_collides

    except ValueError as e:
        # Unusable length/width at the challenger's first valid frame, or an agent
        # with no valid timesteps at all. Counted, never silently dropped.
        key = 'unusable_geometry' if 'unusable' in str(e) else 'never_valid'
        skipped[key] += 1
        if len(skip_examples) < 5:
            skip_examples.append((sid, tgt, key, str(e)[:70]))
        continue

    rows.append({'scenario_id': sid, 'target_idx': tgt, 'target_type': tgt_type,
                 'drift': float(err), 'collides': bool(collides), 'gap': gap})

drift = np.array([r['drift'] for r in rows], dtype=float)
collides = np.array([r['collides'] for r in rows], dtype=bool)
gaps = np.array([r['gap'] for r in rows], dtype=bool)
kinds = np.array([r['target_type'] for r in rows], dtype=int)

print("=" * 70)
print("BASELINE REPLAY DRIFT — full shard, max_baseline_drift=None")
print("=" * 70)
print(f"scenarios measured: {len(rows)}")
for key, n in skipped.items():
    print(f"  skipped, {key:<18s}: {n}")
for sid, tgt, key, msg in skip_examples:
    print(f"    e.g. {sid} agent {tgt} [{key}]: {msg}")

print()
print("-- hard gate (the zero-delta replay ALREADY collides) --")
print("   A separate question from drift, and reported separately: this refusal")
print("   fires regardless of any threshold, so no choice of max_baseline_drift")
print("   changes it.")
if len(rows):
    print(f"   triggered: {int(collides.sum())}/{len(rows)} "
          f"({100.0 * collides.mean():.1f}%)")

def percentiles(label, values):
    if not len(values):
        print(f"   {label:<28s} (no samples)")
        return
    q = np.percentile(values, [50, 75, 90, 95, 99])
    print(f"   {label:<28s} n={len(values):<5d} p50={q[0]:7.3f}  p75={q[1]:7.3f}  "
          f"p90={q[2]:7.3f}  p95={q[3]:7.3f}  p99={q[4]:7.3f}  max={values.max():8.3f}")

print()
print("-- soft-gate drift distribution (metres) --")
print("   Percentiles, not mean/max: a threshold decision needs the SHAPE of the")
print("   tail, and a mean hides it entirely.")
percentiles('all measured', drift)
percentiles('excluding hard-gate rows', drift[~collides])

print()
print("-- where the current 0.5 m default falls --")
if len(drift):
    for thresh in (0.1, 0.25, 0.5, 1.0, 2.0):
        under = float((drift <= thresh).mean())
        mark = '   <- current default' if thresh == 0.5 else ''
        print(f"   drift <= {thresh:4.2f} m : {100 * under:5.1f}% of scenarios{mark}")

print()
print("-- drift by interior validity gap (Batch 1's prediction) --")
print("   Batch 1 reasoned that a rollout across an interior gap joins two")
print("   observations that are not consecutive in time, so drift should")
print("   concentrate there. A refutation is the more interesting result.")
percentiles('with interior gap', drift[gaps])
percentiles('without interior gap', drift[~gaps])

print()
print("-- drift by challenger agent type --")
print("   Vehicles use the bicycle model; pedestrians and cyclists use the linear")
print("   model, which has no heading state. Block 2 Concept 6's fidelity argument")
print("   was derived from vehicle behaviour only.")
for code in sorted(set(kinds.tolist())):
    sel = kinds == code
    percentiles(f"{TYPE_NAMES.get(code, code)} (n={int(sel.sum())})", drift[sel])

print()
print("-- worst 10 by drift --")
print("   A threshold argument built on three pathological outliers is a different")
print("   argument from one built on a fat tail.")
for r in sorted(rows, key=lambda r: -r['drift'])[:10]:
    print(f"   {r['scenario_id']:<24s} agent {r['target_idx']:<4d} "
          f"{TYPE_NAMES.get(r['target_type'], r['target_type']):<10s} "
          f"drift={r['drift']:9.3f} m  gap={r['gap']!s:<5s} collides={r['collides']}")

print()
print("DECISION THIS FEEDS: whether max_baseline_drift stays at 0.5 m.")
print("  bimodal with a clean valley -> put the threshold in the valley")
print("  continuous                  -> any threshold is arbitrary; record drift")
print("                                 per scenario instead of gating on it")
print("  nearly all below 0.5 m      -> the gate is a tripwire, not a filter, and")
print("                                 its value is catching the pathological case")


## 8. Rank + persist

Ranks the real Pass 1 output, prints the top 10, and persists via `upsert_scores`. Then
**upserts the same records a second time** and asserts the row count is unchanged —
idempotency verified against real data, not three synthetic rows.

In [ ]:
ranked = rank_scenarios(records)
print(f"{'rank':<6}{'scenario_id':<40}{'fragility':<12}{'min_ttc':<10}{'min_pet':<10}{'n_agents'}")
for r in ranked[:10]:
    print(f"{r['rank']:<6}{r['scenario_id']:<40}{r['fragility_score']:<12.4f}"
          f"{r['min_ttc']:<10.3f}{r['min_pet']:<10.3f}{r['n_agents']}")

In [ ]:
n1 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_first = cur.fetchone()[0]

n2 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_second = cur.fetchone()[0]

print(f"first upsert:  wrote {n1} rows, table now has {count_after_first}")
print(f"second upsert: wrote {n2} rows, table now has {count_after_second}")
assert count_after_first == count_after_second, (
    "IDEMPOTENCY VIOLATION: row count changed on a repeat upsert of identical records."
)
print("Idempotency confirmed on real data.")

## 9. Pass 2 — `stress_test_scenarios`

Runs the Phase 4 optimizer on the top `TOP_N` scenarios by fragility, with a small DE budget
so this stays fast enough to iterate on. Timed for the Pass-2 cost projection.

In [ ]:
from src.scoring.ranker import top_n_ids
from src.scoring.batch_scorer import stress_test_scenarios

ids_to_test = top_n_ids(records, TOP_N)
print("stress-testing:", ids_to_test)

t0 = time.time()
stress_results = stress_test_scenarios(SHARD_PATH, ids_to_test, de_kwargs=DE_KWARGS,
                                       verbose=True)
pass2_elapsed = time.time() - t0
print(f"\nPass 2 done: {pass2_elapsed:.1f}s for {len(ids_to_test)} scenarios "
      f"({pass2_elapsed/len(ids_to_test):.1f}s/scenario)")

## 10. Pass 2 diagnostics

Per-scenario table plus aggregates: how many hit `no_challenger`/`error`, how many
challengers were non-vehicles (the first real exercise of the linear-model path and of the
DE-only branch, since autograd is vehicle-only), and which perturbation component dominated
each solution — measured in `space.weights`-normalized units so speed (m/s) and heading
(rad) are comparable.

In [ ]:
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
component_names = ["dv0/dvx0", "dtheta0/dvy0", "da_bias/dax_bias", "ddelta_bias/day_bias"]

# re-parse just these scenarios to get types + build a PerturbationSpace for the weights
from src.optimization.perturbation_space import PerturbationSpace

challenger_types = {}
dominant_component = {}
for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in stress_results:
        continue
    r = stress_results[sid]
    if r.get('status') != 'ok' or r.get('target_idx') is None:
        continue
    types_arr = p.get_agent_types()
    tgt = r['target_idx']
    challenger_types[sid] = int(types_arr[tgt])

    space = PerturbationSpace(p.get_agent_states(), p.get_agent_validity(), types_arr,
                              p.get_sdc_index(), tgt)
    weighted = np.abs(np.asarray(r['delta']) * space.weights)
    dominant_component[sid] = component_names[int(np.argmax(weighted))]

print(f"{'scenario_id':<40}{'status':<20}{'collision':<11}{'||delta||':<12}"
      f"{'t_hit':<8}{'target_idx':<12}{'challenger_type':<16}{'dominant'}")
# n_safe WAS n_no_collision'S NAME, and the rename is the finding (audit R08 / A09).
# It counted `status == 'ok' and not collided` — a search that ran to completion
# inside ITS budget, ITS bounds, against ONE heuristically-chosen challenger — and
# printed it as "robustly safe". The API stopped making that claim in Batch 2; this
# cell and the summary cell were never brought along.
#
# replay_infeasible NOW HAS A COUNTER. It had none: it fell through every branch, so
# the aggregate silently did not sum to len(stress_results). Batch 9 makes that worse,
# since scenarios whose logged geometry genuinely overlaps now refuse rather than
# search. The assertion after the loop is what keeps a future status from doing the
# same thing quietly.
n_no_challenger = n_replay_infeasible = n_error = 0
n_collision = n_no_collision = n_non_vehicle = 0
for sid, r in stress_results.items():
    status = r.get('status')
    collided = r.get('collision', False)
    if status == 'ok':
        if collided:
            n_collision += 1
        else:
            n_no_collision += 1
    elif status == 'replay_infeasible':
        n_replay_infeasible += 1
    elif status == 'no_challenger':
        n_no_challenger += 1
    elif status == 'error':
        n_error += 1
    ttype = challenger_types.get(sid)
    ttype_name = type_names.get(ttype, "?") if ttype is not None else "-"
    if ttype is not None and ttype != 1:
        n_non_vehicle += 1
    print(f"{sid:<40}{status:<20}{str(collided):<11}"
          f"{r.get('min_perturbation', float('nan')):<12.4f}"
          f"{r.get('collision_timestep', -1):<8}{r.get('target_idx', -1):<12}"
          f"{ttype_name:<16}{dominant_component.get(sid, '-')}")

print()
print("=" * 70)
print("PASS 2 AGGREGATES")
print("=" * 70)
assert (n_collision + n_no_collision + n_replay_infeasible + n_no_challenger
        + n_error) == len(stress_results), (
    f"the counters sum to "
    f"{n_collision + n_no_collision + n_replay_infeasible + n_no_challenger + n_error}"
    f" but Pass 2 produced {len(stress_results)} results — some status is counted by "
    f"nothing, which is exactly how replay_infeasible went unreported until Batch 10"
)
print(f"no_challenger: {n_no_challenger}   replay_infeasible: {n_replay_infeasible}"
      f"   error: {n_error}   (no search ran for any of these)")
print(f"collision found: {n_collision}   "
      f"no collision found within this search's budget and bounds: {n_no_collision}")
print(f"non-vehicle challengers (types 0/2/3/4, first real linear-model exercise): "
      f"{n_non_vehicle}/{len(challenger_types)}")
type4_or_0 = sum(1 for t in challenger_types.values() if t in (0, 4))
print(f"  of which TYPE_UNSET(0) or TYPE_OTHER(4) specifically: {type4_or_0}")
print(f"seconds/scenario (from cell 9): {pass2_elapsed/max(len(ids_to_test),1):.1f}")
print(f"implied cost of a top-50 stress test: "
      f"{pass2_elapsed/max(len(ids_to_test),1)*50/60:.1f} min")

## 10b. Audit B09 before/after — does the fix ever fire on real data?

Batch 5 changed `refine_scenario` to return the best exact-verified candidate rather than
its final iterate. On the audit's synthetic scene that turned `collision=False` into a
verified collision at norm 0.229. **Nobody knows whether it has ever mattered on real
data**, and the answer decides whether Block 4's description of the refinement stage needs
correcting.

**Its own sample, deliberately not Pass 2's.** `TOP_N = 5`, which after non-vehicle and
replay-infeasible challengers could leave two or three scenarios — too small to support
either conclusion. Raising `TOP_N` was rejected: Pass 2 should run the way the architecture
intends, and inflating it to serve a measurement would corrupt the thing being measured. So
this cell draws `B09_N` scenarios from the ranked Pass 1 output independently. At ~20 s per
vehicle scenario, `B09_N = 25` costs about 8 minutes.

**The pre-fix behaviour has to be reconstructed**, because the B09 fix is baked into
`refine_scenario` and no flag recovers the old path. One loop records both answers, so they
come from the same trajectory by construction rather than by trusting two seeded runs to
coincide.

**The guard on that reconstruction is necessary, not sufficient**, stated rather than
glossed: it compares the reconstruction's best candidate against production's, which proves
nothing whenever the *warm start* wins — and with a DE warm start it usually does.
Mutation-testing confirmed the per-scenario guard stays green under an `lr` change on
real-shaped scenes. So the reconstruction is validated once by an embedded self-test on a
fixture where an *iterate* wins (that one does catch an `lr` change), and the cell reports
how many scenarios the per-scenario guard was actually informative on.

The sample size is printed directly above the conclusion, and a small-sample zero is
reported as "no evidence it fires", never as "evidence it does not".


In [ ]:
import numpy as np
import torch
from src.danger.collision_detector import check_collision_trajectory
from src.optimization.autograd_optimizer import (
    _DiffBicycleRollout, _smooth_margin, refine_scenario,
)
from src.optimization.perturbation_space import (
    PerturbationSpace, ReplayFidelityError, pick_nearest_challenger,
)
from src.optimization.scipy_optimizer import optimize_scenario


def _refine_recording_both(space, delta_init, lam=50.0, lr=0.02, n_iters=300,
                           beta=6.0, n_circles=3, seed=0):
    """
    refine_scenario's loop, reimplemented to record BOTH answers in one pass:

        best   — the smallest exact-verified colliding delta seen (Batch 5 behaviour)
        final  — the last iterate, verified (the PRE-Batch-5 behaviour)

    Reconstructed because there is no flag that recovers the old behaviour: the B09
    fix is baked into refine_scenario, so the only way to measure "what would the old
    code have returned" is to rebuild it. One loop rather than two runs, so `best` and
    `final` come from the same trajectory BY CONSTRUCTION rather than by trusting that
    two seeded runs coincide — and `best` then acts as a fidelity check against
    production, below.
    """
    torch.manual_seed(seed)
    dtype = torch.float64
    roll = _DiffBicycleRollout(space, dtype=dtype)
    delta = torch.tensor(np.asarray(delta_init, np.float64), dtype=dtype,
                         requires_grad=True)
    low = torch.tensor(space.bounds[:, 0], dtype=dtype)
    high = torch.tensor(space.bounds[:, 1], dtype=dtype)
    weights = torch.tensor(space.weights, dtype=dtype)
    opt = torch.optim.Adam([delta], lr=lr)

    def verified(candidate):
        hit, t = check_collision_trajectory(space.apply(candidate), space.validity,
                                            space.sdc_idx, space.target_idx)
        return bool(hit), int(t), space.weighted_norm(candidate)

    best_delta, best_norm, best_from = None, float('inf'), None
    warm = np.asarray(delta_init, np.float32)
    warm_hit, _, warm_norm = verified(warm)
    if warm_hit:
        best_delta, best_norm, best_from = warm.copy(), warm_norm, 'warm_start'

    for it in range(n_iters):
        opt.zero_grad()
        traj = roll.rollout(delta)
        g = _smooth_margin(space, traj, beta, n_circles, dtype)
        loss = ((delta * weights) ** 2).sum() + lam * torch.relu(g)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([delta], max_norm=10.0)
        opt.step()
        with torch.no_grad():
            delta.clamp_(low, high)
        candidate = delta.detach().numpy().astype(np.float32)
        hit, _, norm_c = verified(candidate)
        if hit and norm_c < best_norm:
            best_delta, best_norm, best_from = candidate.copy(), norm_c, f'iterate_{it}'

    final = delta.detach().numpy().astype(np.float32)
    final_hit, _, final_norm = verified(final)
    return {
        'best_delta': best_delta,
        'best_norm': best_norm if best_delta is not None else float('inf'),
        'best_from': best_from,
        'final_collision': final_hit,
        'final_norm': final_norm if final_hit else float('inf'),
    }


# ── embedded self-test: validate the reconstruction ONCE, on a fixture where an
# ITERATE wins ────────────────────────────────────────────────────────────────────
#
# The per-scenario guard below is vacuous whenever the WARM START wins, because the
# warm start's norm is computed identically no matter how wrong the loop is — and
# with a DE warm start it wins most of the time. Measured: perturbing the
# reconstruction's lr or lam left the per-scenario guard green on every real-shaped
# scene tried.
#
# So the reconstruction is validated here instead, against Batch 5's B09 fixture,
# where iterate 12 beats a warm start of norm 0.750. Mutation-tested: an lr change
# breaks this check. It does NOT catch every possible divergence (lam and n_iters
# changes leave the winning candidate unmoved), so this is a necessary condition,
# not a sufficient one.
_st_states = np.zeros((2, 10, 7), dtype=np.float32)
_st_states[:, :, 5:7] = [4.5, 2.0]
_st_states[1, :, 1] = 2.1
_st_validity = np.ones((2, 10), dtype=bool)
_st_space = PerturbationSpace(_st_states, _st_validity, np.ones(2, dtype=int), 0, 1)
_st_warm = np.array([0.0, 0.15, 0.0, 0.0], dtype=np.float32)
_st_prod = refine_scenario(_st_space, delta_init=_st_warm, n_iters=100)
_st_recon = _refine_recording_both(_st_space, _st_warm, n_iters=100)
SELF_TEST_OK = (
    _st_prod['collision'] and _st_recon['best_from'] is not None
    and str(_st_recon['best_from']).startswith('iterate')
    and abs(_st_prod['min_perturbation'] - _st_recon['best_norm']) < 1e-9
)
print(f"reconstruction self-test: {'PASS' if SELF_TEST_OK else 'FAIL'}  "
      f"(production={_st_prod['min_perturbation']:.6f} "
      f"reconstruction={_st_recon['best_norm']:.6f} "
      f"winner={_st_recon['best_from']})")
if not SELF_TEST_OK:
    print("*** The reconstruction does not reproduce production on a fixture where an")
    print("*** iterate wins. Every before/after number below would be meaningless.")
    raise AssertionError('B09 reconstruction self-test failed')

ranked_vehicle = []
for record in ranked:
    if len(ranked_vehicle) >= B09_N:
        break
    ranked_vehicle.append(record['scenario_id'])
wanted = set(ranked_vehicle)

results, infeasible, non_vehicle = [], 0, 0
for raw in ShardLoader(SHARD_PATH):
    if not wanted:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in wanted:
        continue
    wanted.discard(sid)
    states, validity = p.get_agent_states(), p.get_agent_validity()
    types, sdc_idx = p.get_agent_types(), p.get_sdc_index()

    tgt = pick_nearest_challenger(states, validity, sdc_idx)
    if tgt < 0:
        continue
    try:
        space = PerturbationSpace(states, validity, types, sdc_idx, tgt)
    except (ReplayFidelityError, ValueError):
        infeasible += 1
        continue
    if not space.is_vehicle:
        # The autograd path never runs for these — _stress_one gates on
        # `use_autograd and space.is_vehicle` — so B09 cannot apply.
        non_vehicle += 1
        continue

    de = optimize_scenario(space, **DE_KWARGS)
    production = refine_scenario(space, delta_init=de['delta'])
    recon = _refine_recording_both(space, de['delta'])

    # FIDELITY GUARD. The reconstruction above is a reimplementation, and this project
    # does not trust reimplementations — it proves them. Production returns the best
    # verified candidate, which is exactly what `best_norm` is, so the two must agree.
    # If they do not, the reconstruction is not a faithful copy of the old loop and
    # its `final_norm` is meaningless — so every number below is void, not just one.
    faithful = (
        (production['collision'] == (recon['best_delta'] is not None))
        and (not production['collision']
             or abs(production['min_perturbation'] - recon['best_norm']) < 1e-9)
    )
    results.append({
        'scenario_id': sid, 'faithful': faithful,
        # The guard only PROVES anything when an iterate won; if the warm start won,
        # agreement is automatic and says nothing about the loop.
        'guard_informative': str(recon['best_from']).startswith('iterate'),
        'post_collision': production['collision'],
        'post_norm': production['min_perturbation'],
        'pre_collision': recon['final_collision'],
        'pre_norm': recon['final_norm'],
        'winner': recon['best_from'],
    })

print("=" * 70)
print("AUDIT B09 BEFORE/AFTER — does the best-verified fix fire on real data?")
print("=" * 70)
print(f"sample requested (B09_N)        : {B09_N}")
print(f"vehicle challengers evaluated   : {len(results)}")
print(f"  skipped, non-vehicle          : {non_vehicle}")
print(f"  skipped, replay-infeasible    : {infeasible}")

informative = [r for r in results if r['guard_informative']]
print(f"  fidelity guard informative on  : {len(informative)}/{len(results)} "
      f"(only meaningful where an ITERATE won; see the self-test above)")
if results and not informative:
    print("  NOTE: the warm start won on every scenario, so the per-scenario guard")
    print("  proved nothing here. The reconstruction rests on the self-test above")
    print("  plus code inspection.")

unfaithful = [r for r in results if not r['faithful']]
if unfaithful:
    print()
    print(f"*** RECONSTRUCTION MISMATCH on {len(unfaithful)}/{len(results)} scenarios.")
    print("*** The reconstructed loop does not reproduce production's best-verified")
    print("*** answer, so its final-iterate number is not the pre-fix behaviour and")
    print("*** EVERY comparison below is void. Investigate before reading further.")
    for r in unfaithful[:5]:
        print(f"    {r['scenario_id']}: production={r['post_norm']} "
              f"reconstruction_best={r['pre_norm']}")

changed = [r for r in results if r['pre_norm'] != r['post_norm']]
rescued = [r for r in changed if not r['pre_collision'] and r['post_collision']]
improved = [r for r in changed if r['pre_collision'] and r['post_collision']]

print()
print("-- did the fix change the answer? --")
print(f"   identical                       : {len(results) - len(changed)}")
print(f"   changed                         : {len(changed)}")
print(f"     inf -> finite (collision that") 
print(f"       was being discarded)        : {len(rescued)}")
print(f"     finite -> smaller finite      : {len(improved)}")

if improved:
    deltas = np.array([r['pre_norm'] - r['post_norm'] for r in improved], dtype=float)
    print(f"   norm improvement when finite    : "
          f"p50={np.percentile(deltas, 50):.4f}  max={deltas.max():.4f}")

print()
print("-- which candidate won, when the fix mattered --")
print("   The real-data version of the mutation gap Batch 5 found: is the WARM START")
print("   ever the answer, or is it always beaten by some iterate?")
wins = {}
for r in changed:
    key = 'warm_start' if r['winner'] == 'warm_start' else 'an iterate'
    wins[key] = wins.get(key, 0) + 1
for key, n in sorted(wins.items()):
    print(f"   {key:<12s}: {n}")
if not changed:
    print("   (nothing changed, so nothing to attribute)")

print()
print("-- SAMPLE SIZE, stated next to the conclusion --")
n = len(results)
if n == 0:
    print("   NO vehicle challengers evaluated. This cell concluded nothing.")
elif not changed and n < 20:
    print(f"   B09 did not fire on ANY of {n} scenarios — but {n} is a small sample.")
    print(f"   This is 'no evidence it fires', NOT 'evidence it does not'. Raising")
    print(f"   B09_N is the way to strengthen it; at ~20 s per scenario, B09_N=50")
    print(f"   costs about 17 minutes.")
elif not changed:
    print(f"   B09 did not fire on any of {n} scenarios. With n={n} that is real")
    print(f"   evidence the fix is synthetic-only in practice.")
else:
    print(f"   B09 changed {len(changed)} of {n} real results.")

print()
print("DECISION THIS FEEDS: whether B09 needs a Block 4 doctrine note.")
print("  fires often  -> Block 4's description of the refinement stage is materially")
print("                  wrong (it describes a stage that returns its endpoint), and")
print("                  that is a third outstanding textbook correction")
print("  never fires  -> B09 was a correctness fix to a path real data does not")
print("                  exercise; doctrine stands, and THAT is worth recording too")


## 11. `update_stress_results`, then re-read

Writes the Phase 4 columns, then re-reads via `db.fetch_top` and confirms the new columns
landed and the Pass-1 columns (`fragility_score`, `min_ttc`, `min_pet`) were **untouched**.

In [ ]:
n_updated = db.update_stress_results(conn, stress_results)
print(f"updated {n_updated} rows")

top_rows = db.fetch_top(conn, n=TOP_N)
for row in top_rows:
    print(f"{row['scenario_id']:<40} stress_tested_at={row['stress_tested_at']} "
          f"min_perturbation={row['min_perturbation']} "
          f"fragility_score={row['fragility_score']:.4f}")

for row in top_rows:
    orig = next(r for r in records if r['scenario_id'] == row['scenario_id'])
    assert abs(row['fragility_score'] - orig['fragility_score']) < 1e-9, (
        f"Pass-1 fragility_score for {row['scenario_id']} changed after a Pass-2 write "
        f"— update_stress_results must not touch Pass-1 columns."
    )
    # the Phase 3 rework's additive columns must round-trip too
    for col in ('min_ttc', 'min_pet', 'min_ttc_all_pairs', 'min_pet_all_pairs'):
        assert col in row, f"{col} missing from fetch_top row — schema ALTER did not apply"
        if orig.get(col) is not None:
            assert abs(row[col] - orig[col]) < 1e-9, (
                f"{col} for {row['scenario_id']} does not match what Pass 1 wrote"
            )
print("\nConfirmed: Phase 4 columns landed; Pass-1 columns (incl. the new "
      "SDC-restricted and all-pairs diagnostic columns) untouched and round-tripped.")

## 12. Pass 3 — `export_shard_geometry`

**This code has never executed before this cell, on any data, synthetic or real.** It
shipped verified only by import. Prints the full summary dict and every error verbatim.

In [ ]:
from src.scoring.export_geometry import export_shard_geometry

t0 = time.time()
summary = export_shard_geometry(conn, SHARD_PATH, ids_to_test,
                                stress_results=stress_results, verbose=True)
pass3_elapsed = time.time() - t0

print()
print("=" * 70)
print("PASS 3 SUMMARY (export_shard_geometry — first-ever execution)")
print("=" * 70)
print(f"exported:           {summary['exported']}")
print(f"agents_written:     {summary['agents_written']}")
print(f"agents_skipped:     {summary['agents_skipped']}")
print(f"perturbed_written:  {summary['perturbed_written']}")
print(f"errors:             {len(summary['errors'])}")
for e in summary['errors']:
    print(f"  {e}")
print(f"elapsed: {pass3_elapsed:.1f}s")

## 13. Geometry verification — the most important assertion in this notebook

Re-parses the tested scenarios directly and compares the database against ground truth
computed independently in Python.

**The M-array comparison is the key assertion.** With every timestep valid (as in every
synthetic fixture used until now), `ST_M` values equal `0..N-1`, indistinguishable from
plain vertex indices — which is exactly why the original (buggy) exporter design could pass
every prior test. On real data with occlusion, an agent's M sequence should **jump** at a
gap. This cell proves the exporter wrote true timesteps, not vertex positions, by comparing
element-for-element against `np.where(validity[i])[0]`.

In [ ]:
def dump_points_m(conn, table, scenario_id, agent_idx=None):
    with conn.cursor() as cur:
        if table == 'scenario_agents':
            cur.execute("""
                SELECT agent_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM scenario_agents WHERE scenario_id = %s AND agent_idx = %s
            """, (scenario_id, agent_idx))
        else:
            cur.execute("""
                SELECT target_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM perturbed_paths WHERE scenario_id = %s
            """, (scenario_id,))
        return cur.fetchone()

In [ ]:
n_checked = 0
n_exact_match = 0
gap_example_found = False
skip_reconciled = 0

# AUDIT B19. n_exact_match == n_checked is TRUE AT 0 == 0. Every agent below that is
# missing from the database prints MISMATCH and `continue`s WITHOUT incrementing
# n_checked, so a run in which nothing at all was exported satisfied the assertion
# perfectly — the cell whose entire job is catching missing geometry was blind to
# geometry being missing in full. These two track what was SUPPOSED to be checked, so
# the assertion has a denominator that cannot collapse to zero unnoticed.
n_expected = 0     # agents with >= 2 valid timesteps, i.e. genuinely exportable
missing = []       # expected agents with no row in scenario_agents

for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in ids_to_test:
        continue
    validity = p.get_agent_validity()
    N = validity.shape[0]

    with conn.cursor() as cur:
        cur.execute("SELECT agent_idx FROM scenario_agents WHERE scenario_id = %s", (sid,))
        exported_idxs = set(row[0] for row in cur.fetchall())

    for i in range(N):
        true_valid_ts = np.where(validity[i])[0]
        if len(true_valid_ts) < 2:
            skip_reconciled += int(i not in exported_idxs)
            continue
        n_expected += 1
        if i not in exported_idxs:
            missing.append((sid, i, len(true_valid_ts)))
            print(f"MISMATCH: agent {i} in {sid} has {len(true_valid_ts)} valid "
                  f"timesteps but was not exported")
            continue

        row = dump_points_m(conn, 'scenario_agents', sid, i)
        _, npoints, headings, m_values = row
        n_checked += 1

        assert npoints == len(true_valid_ts), (
            f"{sid} agent {i}: ST_NPoints={npoints} != {len(true_valid_ts)} valid timesteps"
        )
        m_array = np.array(m_values, dtype=int)
        if np.array_equal(m_array, true_valid_ts):
            n_exact_match += 1
        else:
            print(f"MISMATCH: {sid} agent {i}: M values {m_array[:10]}... != "
                  f"true valid timesteps {true_valid_ts[:10]}...")

        assert len(headings) == npoints, (
            f"{sid} agent {i}: len(headings)={len(headings)} != ST_NPoints={npoints}"
        )

        # is this a genuine interior-gap agent? report the first one found, verbatim.
        if not gap_example_found and len(true_valid_ts) >= 2:
            span = true_valid_ts[-1] - true_valid_ts[0] + 1
            if span != len(true_valid_ts):
                jump_pos = np.where(np.diff(true_valid_ts) > 1)[0][0]
                print(f"\nINTERIOR GAP EXAMPLE — {sid} agent {i}:")
                print(f"  M sequence around the jump: "
                      f"...{m_array[max(0,jump_pos-2):jump_pos+3]}...")
                print(f"  (jumps from {m_array[jump_pos]} to {m_array[jump_pos+1]}, "
                      f"skipping {m_array[jump_pos+1]-m_array[jump_pos]-1} invalid timesteps)")
                gap_example_found = True

print(f"\nagents expected (>=2 valid timesteps): {n_expected}")
print(f"agents checked: {n_checked}   exact M-array matches: {n_exact_match}")

# Order matters. Assert COVERAGE before correctness, because a vacuous pass is the
# failure this cell exists to catch and the equality below cannot see it.
assert not missing, (
    f"{len(missing)} exportable agent(s) have no row in scenario_agents: "
    f"{missing[:10]}{' ...' if len(missing) > 10 else ''}"
)
assert n_checked == n_expected, (
    f"only {n_checked} of {n_expected} exportable agents were verified — the check "
    f"did not cover everything it was supposed to"
)
assert n_expected > 0, (
    "no exportable agents were found at all; this cell verified nothing and would "
    "otherwise have reported success"
)
assert n_exact_match == n_checked, "Some agents' M arrays did not match true valid timesteps."
print(f"CONFIRMED: all {n_expected} exportable agents present, and the M ordinate "
      f"equals true valid-timestep indices element-for-element.")

if not gap_example_found:
    print("\nNo interior-gap agent found among the tested scenarios' agents — reported")
    print("explicitly rather than silently: this sample happened not to contain one.")

print(f"\nagents with <2 valid timesteps, correctly skipped (not crashed on): "
      f"{skip_reconciled}")
print(f"reconciles against Pass 3's agents_skipped={summary['agents_skipped']} "
      f"(this loop only covers the {len(ids_to_test)} Pass-3 scenarios, so equality is "
      f"expected only if agents_skipped was computed over exactly this set)")

## 14. API check — the full round trip on real data

Starts uvicorn in a daemon thread (modern uvicorn skips installing signal handlers off the
main thread, so `Server.run()` works here) and polls `/health` until ready. `PGUSER` /
`PGPASSWORD` / `PGDATABASE` were exported back in cell 4 — before this is the first import
of `src.api.main`, since `Settings()` builds itself at import time.

The closing assertion is the point of this cell: confirm the `timesteps` array the HTTP
endpoint returns for a real scenario matches the M values read directly from Postgres in
cell 13 — proving the chain shard → exporter → PostGIS → HTTP holds together on real data.

In [ ]:
import threading
import uvicorn
import httpx

from src.api.main import app

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

base = "http://127.0.0.1:8000"
deadline = time.time() + 30
ready = False
last_error = None
while time.time() < deadline:
    try:
        r = httpx.get(f"{base}/health", timeout=1.0)
        if r.status_code == 200:
            ready = True
            break
        last_error = f"HTTP {r.status_code}: {r.text[:200]}"
    except httpx.HTTPError as e:
        # Expected while uvicorn is still binding its socket — connection refused
        # is the normal state for the first second or so. Not silently swallowed:
        # the last one seen is reported below if the deadline expires without success.
        last_error = f"{type(e).__name__}: {e}"
    time.sleep(0.5)

assert ready, (
    f"uvicorn did not become ready within 30s. Last error: {last_error}. "
    f"Check PGUSER/PGPASSWORD/PGDATABASE were exported in cell 4 before this import."
)
print("Server ready.")

In [ ]:
print("GET /health"); print(httpx.get(f"{base}/health").json())
print()
print("GET /stats"); print(httpx.get(f"{base}/stats").json())
print()
print("GET /scenarios?limit=3")
print(httpx.get(f"{base}/scenarios", params={"limit": 3}).json())

# AUDIT R09. This used to be a bare next(...) with no default, so a Pass 2 sample
# containing no verified collision killed the notebook with StopIteration — after
# Pass 1, Pass 2 and Pass 3 had all run. That is not a corner case: Batch 1 made
# `replay_infeasible` a routine outcome, a completed search finding nothing is also
# routine, and TOP_N = 5 makes "none of the five collided" an ordinary result.
# Reproduced against the pre-fix cell with a three-result sample, zero collisions.
stress_tested_sid = next((sid for sid, r in stress_results.items()
                          if r.get('status') == 'ok' and r.get('collision')), None)
COLLISION_AVAILABLE = stress_tested_sid is not None

# Whatever happens, the API checks below still run. Most of them never needed a
# collision: /trajectories wants exported geometry, and /perturbed's score fields
# (min_perturbation, delta, collision_timestep) come from scenario_scores and are
# nullable by design. Only the collision-specific readings are lost.
candidates = [stress_tested_sid] if COLLISION_AVAILABLE else list(ids_to_test)
traj = None
for candidate in candidates:
    probe = httpx.get(f"{base}/scenarios/{candidate}/trajectories").json()
    if probe.get('agents'):
        stress_tested_sid, traj = candidate, probe
        break

assert traj is not None, (
    "no scenario in ids_to_test has exported geometry — Pass 3 exported nothing, "
    "which is a real failure rather than the no-collision case this branch handles"
)

if not COLLISION_AVAILABLE:
    print()
    print("NOTE: no scenario in this Pass 2 sample produced a VERIFIED COLLISION.")
    print("      That is an ordinary outcome, not a failure — a refused replay and a")
    print("      completed search that found nothing are both normal. Outcomes seen:")
    for sid, r in stress_results.items():
        print(f"        {sid}: status={r.get('status')} outcome={r.get('outcome')}")
    print(f"      Falling back to {stress_tested_sid}, which has exported geometry.")
    print("      COLLISION-SPECIFIC CHECKS ARE SKIPPED, not passed: the delta and")
    print("      collision_timestep readings below will be null, and the perturbed")
    print("      path does not exist. Everything else is still exercised.")

print(f"\nGET /scenarios/{stress_tested_sid}/trajectories")
print(f"  {len(traj['agents'])} agents; first agent timesteps[:10] = "
      f"{traj['agents'][0]['timesteps'][:10]}")

print(f"\nGET /scenarios/{stress_tested_sid}/perturbed")
pert = httpx.get(f"{base}/scenarios/{stress_tested_sid}/perturbed").json()
print(f"  delta={pert['delta']}  collision_timestep={pert['collision_timestep']}")
if not COLLISION_AVAILABLE:
    # AUDIT A04, second half. This used to say the values above were null because no
    # collision was found, which is wrong about WHICH values. update_stress_results
    # stores the delta whenever a SEARCH RAN, so a completed unsuccessful search keeps
    # its best candidate delta — measured, one such delta was
    # [2.686, -0.159, 0.458, -0.021]. Only collision_timestep and min_perturbation are
    # necessarily absent, because only those describe a collision that did not happen.
    print("  (collision_timestep is null because no collision was found. delta may")
    print("   still be present: a completed search keeps its best candidate, and only")
    print("   the collision-specific fields are absent.)")


In [ ]:
# close the loop: HTTP timesteps must equal the M values read directly from Postgres
#
# AUDIT R09, second half. This used to read pert['target_idx'] unconditionally — so
# fixing the cell above ALONE would have moved the crash here rather than removed it:
# with no perturbed path target_idx is None, and `next(a for a in traj['agents'] if
# a['agent_idx'] == None)` raises StopIteration in turn. Demonstrated, not assumed.
#
# The round trip does not actually need a collision. It asserts that the timesteps
# served over HTTP equal the M ordinates stored in PostGIS, which is true of ANY
# exported agent — section 13 calls this the most important assertion in the
# notebook, so it runs either way rather than being skipped for want of a delta.
# AUDIT A04. Chosen by MEMBERSHIP IN THE EXPORTED SET, not by trusting target_idx to
# name an agent that was exported. R06 persists target_idx on scenario_scores even when
# B15 declined to export that agent — an agent with fewer than two valid frames cannot
# form a LINESTRING, so it is skipped — and the guard below used to test only for
# target_idx IS None. A scenario with a real collision, a real target_idx and no
# geometry for that specific agent therefore reached a bare next(...) and raised
# StopIteration. Measured end to end: written=1 skipped=1, exported agent set [0],
# target_idx=1, under BOTH a verified collision and a completed no-collision search.
#
# NOT folded into the R09 fallback in the cell above, and the reason is granularity:
# that one selects a SCENARIO with exported geometry, this selects an AGENT within the
# scenario already chosen. Merging them would discard a scenario whose round-trip
# assertion runs perfectly well on a different agent.
exported_agents = {a['agent_idx']: a for a in traj['agents']}
target_idx_api = pert['target_idx']
if target_idx_api is None:
    target_idx_api = next(iter(exported_agents))
    print(f"no perturbed path for {stress_tested_sid}; closing the round trip on "
          f"agent {target_idx_api} instead — the assertion is identical")
elif target_idx_api not in exported_agents:
    missing_challenger = target_idx_api
    target_idx_api = next(iter(exported_agents))
    print(f"challenger {missing_challenger} of {stress_tested_sid} has no exported "
          f"geometry (fewer than two valid frames, so B15 skipped it); closing the "
          f"round trip on agent {target_idx_api} instead — the assertion is identical")

http_agent = exported_agents[target_idx_api]
http_timesteps = np.array(http_agent['timesteps'])

row = dump_points_m(conn, 'scenario_agents', stress_tested_sid, target_idx_api)
_, npoints_db, _, m_values_db = row
db_m = np.array(m_values_db, dtype=float)

assert np.array_equal(http_timesteps, db_m), (
    "HTTP /trajectories timesteps do not match the M values read directly from Postgres — "
    "the shard-to-HTTP round trip is broken somewhere between cells 12 and 14."
)
print("CONFIRMED: HTTP timesteps == Postgres M values. Full round trip closed on real data.")
if not COLLISION_AVAILABLE:
    print("  (closed on a non-colliding scenario — see the note above)")

server.should_exit = True


## 15. Summary

Self-contained — safe to copy out of the notebook on its own. Leads with the two numbers
that confirm the Phase 3 rework did what it was meant to.

In [ ]:
print("=" * 70)
print("COLAB VALIDATION RUN — SUMMARY  (post Phase 3 rework)")
print("=" * 70)
print()
print("-- Did the rework work? --")
print(f"[Finding 1] old all-pairs ranking vs new SDC-restricted ranking: "
      f"Spearman rho={rho_all_pairs_vs_sdc:.4f}, top-{TOP_N} overlap={len(topN_all & topN_sdc)}/{TOP_N}")
print(f"            (low rho / small overlap = the rework changed the ranking, as intended;")
print(f"             validation baseline was rho=-0.0152, 2/5 overlap)")
print(f"[Finding 1] TTC floor-saturation: SDC-restricted {100*ttc_sdc_cap.mean():.1f}%  "
      f"vs all-pairs {100*ttc_all_cap.mean():.1f}%")
print(f"[Finding 5] negative SDC min_pet, all confirmed genuine overlaps: "
      f"{n_negative} pair(s), 0 sign-semantics regressions")
print()
print("-- Pass 1 --")
print(f"scenarios scored: {len(records)}   errored: {len(errors)}")
print(f"fragility range: [{frag.min():.4f}, {frag.max():.4f}]  mean={frag.mean():.4f}")
print(f"SDC-restricted min_ttc floor-capped: {100*ttc_sdc_cap.mean():.1f}%   "
      f"(all-pairs diagnostic: {100*ttc_all_cap.mean():.1f}%)")
print(f"SDC-restricted min_pet floor-capped: {100*pet_sdc_cap.mean():.1f}%   "
      f"negative: {100*pet_sdc_negative.mean():.1f}%")
print(f"agent tracks with an interior validity gap: {interior_gap}/{total_tracks}")
print()
print("-- Pass 2 --")
print(f"stress-tested: {len(ids_to_test)}   collisions found: {n_collision}")
# NOT "robustly safe" (audit R08 / A09) — see the counter comment in the Pass 2
# diagnostics cell. A completed search that found nothing is a statement about the
# search; a refused replay is a statement that no search happened at all.
print(f"no collision found within this search's budget and bounds: {n_no_collision}")
print(f"replay refused, no search ran: {n_replay_infeasible}   "
      f"no challenger: {n_no_challenger}   errored: {n_error}")
print(f"non-vehicle challengers: {n_non_vehicle}/{len(challenger_types)}")
print()
print("-- Pass 3 (export_shard_geometry — first execution ever) --")
print(f"exported: {summary['exported']}   agents_written: {summary['agents_written']}   "
      f"agents_skipped: {summary['agents_skipped']}   errors: {len(summary['errors'])}")
print(f"geometry M-ordinate verification: "
      f"{'PASSED' if n_exact_match == n_checked else 'FAILED'} "
      f"({n_exact_match}/{n_checked} agents)")
print()
print("-- Wall clock --")
print(f"Pass 1: {pass1_elapsed:.1f}s   Pass 2: {pass2_elapsed:.1f}s   "
      f"Pass 3: {pass3_elapsed:.1f}s")
print(f"total notebook compute (excludes installs/mount): "
      f"{pass1_elapsed + pass2_elapsed + pass3_elapsed:.1f}s")